# Dashboard 模型运行台账集成 v1

目标：验证 dashboard export/view model 能通过 manifest 读取 `model_run_ledger`，并把每个模型的训练、回测、analysis、manifest 状态暴露到页面交互层。

## 实验契约

- Baseline：`research/quantbench_all_model_run_ledger_v1` 全模型矩阵 card。
- Role：diagnose，验证模型运行台账从 `src/model` sidecar 进入 `src/dashboard` payload/view model。
- Changed layer：dashboard。
- Acceptance：dashboard payload 包含 `model_run_coverage` 和 `model_run_ledger`；view model 的每个 top model card 包含 `run_record` 与 `run_status`。
- Failure mode：dashboard 只展示收益/regime 表，无法判断模型是否完成训练、回测、objective、promotion、analysis 和 manifest 记录。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').exists() and (path / 'AGENTS.md').exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.dashboard import build_dashboard_view_model, write_model_matrix_dashboard_export

CARD_DIR = repo_root / 'research' / 'dashboard_model_run_ledger_integration_v1'
SOURCE_CARD = repo_root / 'research' / 'quantbench_all_model_run_ledger_v1'
DASHBOARD_EXPORT = CARD_DIR / 'dashboard_export.json'
VIEW_MODEL_EXPORT = CARD_DIR / 'view_model.json'
SOURCE_CARD.exists(), DASHBOARD_EXPORT

(True,
 PosixPath('/Users/rziiiii/dev/workspace/research/dashboard_model_run_ledger_integration_v1/dashboard_export.json'))

In [2]:
payload = write_model_matrix_dashboard_export(SOURCE_CARD, output_path=DASHBOARD_EXPORT, top_n=12)
view = build_dashboard_view_model(payload)
VIEW_MODEL_EXPORT.write_text(json.dumps(view, ensure_ascii=False, indent=2), encoding='utf-8')

coverage = payload['model_run_coverage']
ledger = pd.DataFrame(payload['model_run_ledger'])
cards = pd.DataFrame([
    {
        'model': card['model'],
        'run_status': card['run_status'],
        'trained': card['run_record'].get('trained'),
        'backtested': card['run_record'].get('backtested'),
        'analysis_recorded': card['run_record'].get('analysis_recorded'),
        'manifested': card['run_record'].get('manifested'),
        'paper_fidelity': card['paper_fidelity'],
    }
    for card in view['model_cards']
])
cards.to_csv(CARD_DIR / 'view_model_run_cards.csv', index=False)
coverage

{'schema_version': 'model.run_ledger_summary.v1',
 'expected_model_count': 32,
 'ledger_model_count': 32,
 'registered_model_count': 32,
 'trained_model_count': 32,
 'predicted_model_count': 32,
 'backtested_model_count': 32,
 'objective_evaluated_model_count': 32,
 'promotion_evaluated_model_count': 32,
 'analysis_recorded_model_count': 32,
 'manifested_model_count': 32,
 'proxy_model_count': 17,
 'native_model_count': 15,
 'paper_fidelity_counts': {'native_library': 8,
  'native_sequence': 2,
  'native_torch': 5,
  'proxy_feature_adapter': 17},
 'adapter_statuses': ['paper_strict_adapter'],
 'missing_models': [],
 'all_models_completed': True}

In [3]:
summary_rows = {
    'payload_schema': payload['schema_version'],
    'view_schema': view['schema_version'],
    'ledger_rows': len(payload['model_run_ledger']),
    'card_rows': len(view['model_cards']),
    'all_models_completed': coverage.get('all_models_completed'),
    'trained_model_count': coverage.get('trained_model_count'),
    'backtested_model_count': coverage.get('backtested_model_count'),
    'missing_models': coverage.get('missing_models'),
    'adapter_statuses': view['controls'].get('adapter_statuses'),
    'run_statuses': view['controls'].get('run_statuses'),
}
summary_rows

{'payload_schema': 'dashboard.model_matrix.v1',
 'view_schema': 'dashboard.view_model.v1',
 'ledger_rows': 32,
 'card_rows': 12,
 'all_models_completed': True,
 'trained_model_count': 32,
 'backtested_model_count': 32,
 'missing_models': [],
 'adapter_statuses': ['paper_strict_adapter'],
 'run_statuses': ['complete']}

In [4]:
assert payload['model_run_coverage']['all_models_completed'] is True
assert len(payload['model_run_ledger']) == 32
assert 'model_run_ledger' in payload['artifacts']
assert 'model_run_ledger_summary' in payload['artifacts']
assert view['model_run_coverage']['schema_version'] == 'model.run_ledger_summary.v1'
assert view['controls']['run_statuses'] == ['complete']
assert view['model_cards']
assert all(card['run_status'] == 'complete' for card in view['model_cards'])
assert all(card['run_record'].get('trained') for card in view['model_cards'])
assert all(card['run_record'].get('backtested') for card in view['model_cards'])
assert all(card['run_record'].get('analysis_recorded') for card in view['model_cards'])
assert all(card['run_record'].get('manifested') for card in view['model_cards'])
'dashboard_model_run_ledger_integration_passed'

'dashboard_model_run_ledger_integration_passed'

In [5]:
from IPython.display import Markdown

notebook_summary = {
    'schema_version': 'research.dashboard_model_run_ledger_integration_v1',
    'source_card': str(SOURCE_CARD.relative_to(repo_root)),
    'dashboard_export': 'dashboard_export.json',
    'view_model_export': 'view_model.json',
    'view_model_run_cards': 'view_model_run_cards.csv',
    **summary_rows,
}
(CARD_DIR / 'notebook_summary.json').write_text(json.dumps(notebook_summary, ensure_ascii=False, indent=2), encoding='utf-8')

conclusion = f'''## 中文结论

- 做了什么：从 `{SOURCE_CARD.relative_to(repo_root)}` 读取全模型矩阵 manifest，生成 dashboard export 和 view model。
- 核心指标：ledger_rows={summary_rows['ledger_rows']}，card_rows={summary_rows['card_rows']}，trained={summary_rows['trained_model_count']}，backtested={summary_rows['backtested_model_count']}，all_models_completed={summary_rows['all_models_completed']}。
- 页面交互状态：view model controls 暴露 adapter_statuses={summary_rows['adapter_statuses']}，run_statuses={summary_rows['run_statuses']}；每个 model card 带 `run_record` 和 `run_status`。
- Regime/区间差异：dashboard payload 仍通过 `analysis_overview`、`segment_effect_matrix`、`model_review_briefing` 展示模型差异；本次新增的是运行覆盖证明，避免页面只展示收益结果而看不到模型是否完整进入链路。
- 是否晋升：本次不晋升生产模型，只确认 dashboard 可以审阅全模型训练回测完整性。
- 风险：如果后续矩阵 card 没有写 `model_run_ledger`，dashboard 会显示 unknown/空 coverage，应优先修复矩阵产物而不是在页面层猜测。
'''
Markdown(conclusion)

## 中文结论

- 做了什么：从 `research/quantbench_all_model_run_ledger_v1` 读取全模型矩阵 manifest，生成 dashboard export 和 view model。
- 核心指标：ledger_rows=32，card_rows=12，trained=32，backtested=32，all_models_completed=True。
- 页面交互状态：view model controls 暴露 adapter_statuses=['paper_strict_adapter']，run_statuses=['complete']；每个 model card 带 `run_record` 和 `run_status`。
- Regime/区间差异：dashboard payload 仍通过 `analysis_overview`、`segment_effect_matrix`、`model_review_briefing` 展示模型差异；本次新增的是运行覆盖证明，避免页面只展示收益结果而看不到模型是否完整进入链路。
- 是否晋升：本次不晋升生产模型，只确认 dashboard 可以审阅全模型训练回测完整性。
- 风险：如果后续矩阵 card 没有写 `model_run_ledger`，dashboard 会显示 unknown/空 coverage，应优先修复矩阵产物而不是在页面层猜测。
